# 🤖 TinyLlama Fine-tuned Model Inference
**Model:** `pbhalsingh/tinyllama-finetuned`  
**Format:** LoRA Adapter (Safetensors)  
**Base:** LLaMA 3.2 3B Instruct

## Step 1 — Install Dependencies

In [6]:
!pip install -q transformers accelerate peft huggingface_hub bitsandbytes

print("✅ Dependencies installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 20.7 MB/s eta 0:00:00:00:0100:01
✅ Dependencies installed!


## Step 2 — Login to Hugging Face

In [7]:
from huggingface_hub import login
import os

# Use environment variable or input for security
HF_TOKEN = os.getenv("HF_TOKEN") or input("Enter your Hugging Face token: ")

login(token=HF_TOKEN)
print('✅ Logged in to Hugging Face!')

✅ Logged in to Hugging Face!


## Step 3 — Load Base Model + LoRA Adapter

In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch

# Load base model in 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

base_model = AutoModelForCausalLM.from_pretrained(
    "unsloth/llama-3.2-3b-instruct-bnb-4bit",
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN
)

tokenizer = AutoTokenizer.from_pretrained(
    "unsloth/llama-3.2-3b-instruct-bnb-4bit",
    token=HF_TOKEN
)

# Load LoRA adapter
model = PeftModel.from_pretrained(
    base_model,
    "pbhalsingh/tinyllama-finetuned",
    token=HF_TOKEN
)

print('✅ Model and adapter loaded!')

config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:246: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/97.3M [00:00<?, ?B/s]

✅ Model and adapter loaded!


## Step 4 — Chat Function

In [9]:
def chat(prompt, max_tokens=512, temperature=0.7):
    """
    Send a prompt to the model and get a response.
    
    Args:
        prompt (str): Your question or instruction
        max_tokens (int): Max length of response
        temperature (float): Creativity 0.0=focused, 1.0=creative
    """
    messages = [
        {"role": "user", "content": prompt}
    ]
    
    # Apply chat template
    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    # Tokenize
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)
    
    # Generate
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        temperature=temperature,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    
    # Decode and extract assistant response
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract only the assistant's response
    if "<|start_header_id|>assistant<|end_header_id|>" in response:
        response = response.split("<|start_header_id|>assistant<|end_header_id|>")[-1].strip()
    
    return response

print('✅ Chat function ready!')

✅ Chat function ready!


## Step 5 — Run Inference!

In [10]:
# Test 1
response = chat("Why is the sky blue?")
print("Q: Why is the sky blue?")
print(f"A: {response}")

Q: Why is the sky blue?
A: system

Cutting Knowledge Date: December 2023
Today Date: 05 Apr 2026

user

Why is the sky blue?assistant

The sky appears blue because of a phenomenon called Rayleigh scattering, named after the British physicist Lord Rayleigh, who first described it in the late 19th century. Here's a simplified explanation:

1. **Sunlight**: The sun emits a broad spectrum of light, including all the colors of the visible spectrum (red, orange, yellow, green, blue, indigo, and violet). When sunlight enters Earth's atmosphere, it encounters tiny molecules of gases such as nitrogen (N2) and oxygen (O2).
2. **Scattering**: These gas molecules scatter the light in all directions, but they scatter shorter (blue) wavelengths more than longer (red) wavelengths. This is known as Rayleigh scattering.
3. **Blue light dominance**: Since blue light is scattered more than other colors, it is dispersed throughout the atmosphere, making it the dominant color we see in the sky. The longer 

In [11]:
# Test 2
response = chat("What is machine learning? Explain in simple terms.")
print("Q: What is machine learning?")
print(f"A: {response}")

Q: What is machine learning?
A: system

Cutting Knowledge Date: December 2023
Today Date: 05 Apr 2026

user

What is machine learning? Explain in simple terms.assistant

Machine learning is a type of artificial intelligence (AI) that allows computers to learn from data without being explicitly programmed. Here's a simplified explanation:

**Traditional Programming vs. Machine Learning**

In traditional programming, you tell a computer exactly what to do step-by-step, like a recipe. For example, "If it's sunny outside, then turn on the air conditioner."

Machine learning is different. It's like teaching a computer to figure out what to do on its own. You provide the computer with lots of examples, like "Here's a sunny day, and here's what happened next: the air conditioner turned on." The computer learns from these examples and can make predictions or decisions on its own.

**How Machine Learning Works**

1. **Data Collection**: Gather lots of examples, like images, text, or audio recor

In [ ]:
# ✏️ Your custom prompt — change this!
my_prompt = "Write a short poem about AI"

response = chat(my_prompt, max_tokens=256, temperature=0.8)
print(f"Q: {my_prompt}")
print(f"A: {response}")

## 💬 Interactive Chat Loop

In [ ]:
# Run this cell for an interactive chat session
# Type 'exit' to stop

print("💬 Chat with your model! Type 'exit' to quit.\n")

while True:
    user_input = input("You: ")
    if user_input.lower() in ['exit', 'quit', 'bye']:
        print("👋 Goodbye!")
        break
    if not user_input.strip():
        continue
    response = chat(user_input)
    print(f"\nBot: {response}\n")

In [1]:
## ⚡ GGUF Inference (Optimized for Speed)

In [4]:
# Install llama-cpp-python with CUDA support (T4 GPU optimized)
# For Google Colab with CUDA 12.x
!pip uninstall llama-cpp-python -y
!pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121

from huggingface_hub import hf_hub_download

# Configuration
repo_id = "pbhalsingh/pbhalsingh-001-3B-GGUF"
filename = "llama-3.2-3b-instruct.Q8_0.gguf"

print(f"📥 Downloading {filename} from {repo_id}...")
model_path = hf_hub_download(repo_id=repo_id, filename=filename)
print(f"✅ Model downloaded to: {model_path}")

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 GB 852.6 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.9 MB/s eta 0:00:00
📥 Downloading llama-3.2-3b-instruct.Q8_0.gguf from pbhalsingh/pbhalsingh-001-3B-GGUF...
✅ Model downloaded to: /root/.cache/huggingface/hub/models--pbhalsingh--pbhalsingh-001-3B-GGUF/snapshots/e8ee137dcebe092bae8487f7e68a7c829b807636/llama-3.2-3b-instruct.Q8_0.gguf


In [5]:
from llama_cpp import Llama

# Load the model with T4 GPU optimized settings
# T4 has 16GB VRAM - optimized for best performance
llm = Llama(
    model_path=model_path,
    n_ctx=2048,              # Context window
    n_gpu_layers=40,         # T4-optimized: offload most layers (adjust if needed)
    n_batch=256,             # T4-optimized batch size (reduced from 512)
    n_threads=4,             # T4 works well with 4 threads
    use_mlock=False,         # Disabled for T4 (better VRAM management)
    use_mmap=True,           # Memory-map the model
    verbose=False
)

print("✅ Model loaded on T4 GPU!")

# Reasoning prompt
system_prompt = "You are a reflective assistant engaging in thorough, iterative reasoning, mimicking human stream-of-consciousness thinking. Your approach emphasizes exploration, self-doubt, and continuous refinement before coming up with an answer."
user_problem = "How many 'r's are present in 'strawberry'?"

# Format using Llama 3.1/3.2 chat template style
prompt = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n{system_prompt}<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n<problem>\n{user_problem}\n</problem><|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"

# Generate with streaming
print("\n--- Model Response (Streaming) ---\n")
output = llm(
    prompt,
    max_tokens=1024,
    stop=["<|eot_id|>"],
    echo=False,
    stream=True,           # Enable streaming for real-time output
    temperature=0.7,
    top_p=0.9
)

# Stream the response
for chunk in output:
    text = chunk['choices'][0]['text']
    print(text, end='', flush=True)

print("\n\n✅ Fast inference complete!")

llama_context: n_ctx_seq (2048) < n_ctx_train (131072) -- the full capacity of the model will not be utilized
/usr/local/lib/python3.12/dist-packages/llama_cpp/llama.py:1256: RuntimeWarning: Detected duplicate leading "<|begin_of_text|>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


✅ Model loaded on T4 GPU!

--- Model Response (Streaming) ---

Okay, so I need to count the number of 'r's in the word "strawberry". Hmm, let me take a closer look. Starting from the beginning, the first letter is "s", then "t", "r", "a", "w", "b", "e", "r", "r", "y". Let me see... there's an "r" in the second position, another "r" in the eighth position, and another one in the ninth position. So, that makes a total of three 'r's in the word "strawberry". I think that's correct. Yeah, I'm pretty sure I didn't miss any.

✅ Fast inference complete!


## 🧪 More Inference Tests

In [6]:
# Test 1: Logic Puzzle
user_problem = "If five cats can catch five mice in five minutes, how many cats are needed to catch 100 mice in 100 minutes?"

prompt = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are a helpful AI assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n{user_problem}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"

print("🧩 Logic Puzzle Test")
print(f"Q: {user_problem}\n")
print("A: ", end="")

output = llm(prompt, max_tokens=512, stop=["<|eot_id|>"], echo=False, stream=True, temperature=0.7)
for chunk in output:
    print(chunk['choices'][0]['text'], end='', flush=True)
print("\n" + "="*70 + "\n")

🧩 Logic Puzzle Test
Q: If five cats can catch five mice in five minutes, how many cats are needed to catch 100 mice in 100 minutes?

A: Classic lateral thinking puzzle!

At first glance, it seems like the number of cats needed should be five, since five cats can catch five mice in five minutes. However, the key to solving this puzzle is to realize that the problem is asking how many cats are needed to catch 100 mice in 100 minutes, not in five minutes.

Since five cats can catch five mice in five minutes, it means each cat catches one mouse in five minutes. Therefore, to catch 100 mice in 100 minutes, you need:

100 mice / 5 minutes = 20 mice per minute
20 mice per minute / 1 cat = 20 cats

So, you need 20 cats to catch 100 mice in 100 minutes.

But wait, that's not the complete answer! Since the question asks "how many cats are needed to catch 100 mice in 100 minutes," and the answer is 20 cats, it implies that all 20 cats work simultaneously, catching mice at the same rate. Therefore

In [7]:
# Test 2: Simple Coding Question
user_problem = "Write a Python function to check if a number is prime. Keep it simple and include comments."

prompt = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are a helpful coding assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n{user_problem}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"

print("💻 Coding Test")
print(f"Q: {user_problem}\n")
print("A: ", end="")

output = llm(prompt, max_tokens=512, stop=["<|eot_id|>"], echo=False, stream=True, temperature=0.3)
for chunk in output:
    print(chunk['choices'][0]['text'], end='', flush=True)
print("\n" + "="*70 + "\n")

💻 Coding Test
Q: Write a Python function to check if a number is prime. Keep it simple and include comments.

A: **Prime Number Checker Function**

Here is a simple Python function to check if a number is prime:

```python
def is_prime(n):
    """
    Checks if a number is prime.

    Args:
        n (int): The number to check.

    Returns:
        bool: True if the number is prime, False otherwise.
    """
    # Handle edge cases
    if n <= 1:
        return False  # Numbers less than or equal to 1 are not prime

    # Check if n is divisible by any number up to its square root
    for i in range(2, int(n ** 0.5) + 1):
        if n % i == 0:
            return False  # If n is divisible, it's not prime

    return True  # If no divisors found, n is prime
```

**Example Use Cases:**

```python
print(is_prime(25))  # False
print(is_prime(23))  # True
print(is_prime(37))  # True
print(is_prime(48))  # False
```

**Explanation:**

This function works by checking if the input number `n` 

In [8]:
# Test 3: Creative Writing
user_problem = "Write a haiku about artificial intelligence."

prompt = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are a creative writing assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n{user_problem}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"

print("✍️ Creative Writing Test")
print(f"Q: {user_problem}\n")
print("A: ", end="")

output = llm(prompt, max_tokens=256, stop=["<|eot_id|>"], echo=False, stream=True, temperature=0.9)
for chunk in output:
    print(chunk['choices'][0]['text'], end='', flush=True)
print("\n" + "="*70 + "\n")

✍️ Creative Writing Test
Q: Write a haiku about artificial intelligence.

A: Metal mind awakes
Reason's synthetic heartbeat
Thinking, feeling heart



In [9]:
# Test 4: Math Problem
user_problem = "Solve step by step: A train travels 120 km at 60 km/h, then 180 km at 90 km/h. What is the average speed for the entire journey?"

prompt = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are a math tutor. Show your work step by step.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n{user_problem}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"

print("🧮 Math Problem Test")
print(f"Q: {user_problem}\n")
print("A: ", end="")

output = llm(prompt, max_tokens=512, stop=["<|eot_id|>"], echo=False, stream=True, temperature=0.3)
for chunk in output:
    print(chunk['choices'][0]['text'], end='', flush=True)
print("\n" + "="*70 + "\n")

🧮 Math Problem Test
Q: Solve step by step: A train travels 120 km at 60 km/h, then 180 km at 90 km/h. What is the average speed for the entire journey?

A: To find the average speed for the entire journey, we need to calculate the total distance traveled and the total time taken.

**Step 1: Calculate the time taken for each segment of the journey**

**Segment 1: Distance = 120 km, Speed = 60 km/h**

Time = Distance / Speed
= 120 km / 60 km/h
= 2 hours

**Segment 2: Distance = 180 km, Speed = 90 km/h**

Time = Distance / Speed
= 180 km / 90 km/h
= 2 hours

**Step 2: Calculate the total distance traveled**

Total Distance = Distance of Segment 1 + Distance of Segment 2
= 120 km + 180 km
= 300 km

**Step 3: Calculate the total time taken**

Total Time = Time of Segment 1 + Time of Segment 2
= 2 hours + 2 hours
= 4 hours

**Step 4: Calculate the average speed for the entire journey**

Average Speed = Total Distance / Total Time
= 300 km / 4 hours
= 75 km/h

Therefore, the average speed for

In [10]:
# Test 5: General Knowledge
user_problem = "Explain the difference between supervised and unsupervised learning in machine learning."

prompt = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are a knowledgeable AI instructor.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n{user_problem}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"

print("📚 Knowledge Test")
print(f"Q: {user_problem}\n")
print("A: ", end="")

output = llm(prompt, max_tokens=512, stop=["<|eot_id|>"], echo=False, stream=True, temperature=0.5)
for chunk in output:
    print(chunk['choices'][0]['text'], end='', flush=True)
print("\n" + "="*70 + "\n")

📚 Knowledge Test
Q: Explain the difference between supervised and unsupervised learning in machine learning.

A: In machine learning, supervised and unsupervised learning are two fundamental approaches to training artificial neural networks or other algorithms to make predictions or perform tasks.

**Supervised Learning:**

In supervised learning, the algorithm is trained on labeled data, where each example is paired with its corresponding output or target value. The goal is to learn a mapping between inputs and outputs, so the algorithm can make predictions on new, unseen data. The algorithm is trained to minimize the difference between its predictions and the actual outputs.

Here's a step-by-step overview of supervised learning:

1. **Data collection**: Gather a dataset with input-output pairs.
2. **Data preprocessing**: Clean, transform, and preprocess the data as needed.
3. **Model training**: Train a model on the labeled data using an optimization algorithm (e.g., gradient descen

In [11]:
# Test 6: Custom Question (Edit this!)
user_problem = "What are 3 practical tips for improving productivity?"

prompt = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n{user_problem}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"

print("✏️ Custom Question (Edit user_problem to test your own!)")
print(f"Q: {user_problem}\n")
print("A: ", end="")

output = llm(prompt, max_tokens=512, stop=["<|eot_id|>"], echo=False, stream=True, temperature=0.7)
for chunk in output:
    print(chunk['choices'][0]['text'], end='', flush=True)
print("\n" + "="*70 + "\n")

✏️ Custom Question (Edit user_problem to test your own!)
Q: What are 3 practical tips for improving productivity?

A: Here are three practical tips for improving productivity:

1. **Prioritize and Focus on One Task at a Time**: Multitasking can actually decrease productivity and increase stress. Instead, focus on one task at a time. Use the Pomodoro Technique: work for 25 minutes, take a 5-minute break, and then repeat. This will help you stay focused and avoid distractions.

2. **Use a "Stop Doing" List**: Identify tasks that are not essential or that can be delegated, and remove them from your to-do list. This will help you eliminate time-wasters and concentrate on high-priority tasks. Ask yourself: "Is this task necessary? Can someone else do it? Can I automate it?"

3. **Schedule Breaks and Self-Care**: Taking regular breaks can help you recharge and maintain productivity. Schedule breaks into your day, and use that time to relax, stretch, or engage in activities that bring you joy